# Runtime and GPU check

Select your Colab kernel in VS Code, then run all cells. The output describes the selected kernel's machine. A successful GPU check lists an NVIDIA GPU and, when PyTorch is installed with CUDA support, completes a small calculation on it.

In [1]:
import platform
import sys
from pathlib import Path

print('Python:', sys.version)
print('System:', platform.platform())
print('Working directory:', Path.cwd())
print('Kernel connection works.')

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
System: Linux-6.6.122+-x86_64-with-glibc2.39
Working directory: /content
Kernel connection works.


In [2]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
        capture_output=True, text=True, timeout=30,
    )
    print(result.stdout or result.stderr)
    if result.returncode:
        print('GPU inspection failed; check the selected runtime.')
else:
    print('nvidia-smi is unavailable. Check that you selected a GPU runtime.')

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07



In [ ]:
try:
    import torch
except ModuleNotFoundError as exc:
    if exc.name != 'torch':
        raise
    print('PyTorch is not installed in this kernel; skipping the tensor check.')
else:
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        values = torch.tensor([1.0, 2.0, 3.0], device='cuda')
        result = (values * values).sum().item()
        assert result == 14.0, f'Unexpected result: {result}'
        print('GPU calculation passed:', result)
    else:
        print('GPU calculation skipped. Check the runtime and PyTorch CUDA support.')

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU calculation passed: 14.0


In [ ]:
from datasets import Dataset, DatasetDict

def short_value(value):
    # Avoid printing image data itself
    if hasattr(value, "mode") and hasattr(value, "size"):
        return f"<image: mode={value.mode}, size={value.size}>"

    text = repr(value)
    return text if len(text) <= 300 else text[:300] + " ..."

found_data = False

for name, value in list(globals().items()):
    if isinstance(value, DatasetDict):
        found_data = True
        print(f"\nVariable: {name} (DatasetDict)")
        print("Splits:", list(value.keys()))

        if "train" in value:
            train_split = value["train"]
            print("Train rows:", len(train_split))
            print("Train columns:", train_split.column_names)

            row = train_split[0]
            print("First training example:")
            for key, item in row.items():
                print(f"  {key}: {short_value(item)}")

    elif isinstance(value, Dataset):
        found_data = True
        print(f"\nVariable: {name} (Dataset)")
        print("Rows:", len(value))
        print("Columns:", value.column_names)

        row = value[0]
        print("First example:")
        for key, item in row.items():
            print(f"  {key}: {short_value(item)}")

if not found_data:
    print("No Hugging Face Dataset or DatasetDict found in the notebook.")
    print("Tell me what variable you used to load SLAKE, or paste your data-loading cell.")

: 

In [1]:
from datasets import Dataset, DatasetDict

def short_value(value):
    # Avoid printing image data itself
    if hasattr(value, "mode") and hasattr(value, "size"):
        return f"<image: mode={value.mode}, size={value.size}>"

    text = repr(value)
    return text if len(text) <= 300 else text[:300] + " ..."

found_data = False

for name, value in list(globals().items()):
    if isinstance(value, DatasetDict):
        found_data = True
        print(f"\nVariable: {name} (DatasetDict)")
        print("Splits:", list(value.keys()))

        if "train" in value:
            train_split = value["train"]
            print("Train rows:", len(train_split))
            print("Train columns:", train_split.column_names)

            row = train_split[0]
            print("First training example:")
            for key, item in row.items():
                print(f"  {key}: {short_value(item)}")

    elif isinstance(value, Dataset):
        found_data = True
        print(f"\nVariable: {name} (Dataset)")
        print("Rows:", len(value))
        print("Columns:", value.column_names)

        row = value[0]
        print("First example:")
        for key, item in row.items():
            print(f"  {key}: {short_value(item)}")

if not found_data:
    print("No Hugging Face Dataset or DatasetDict found in the notebook.")
    print("Tell me what variable you used to load SLAKE, or paste your data-loading cell.")

No Hugging Face Dataset or DatasetDict found in the notebook.
Tell me what variable you used to load SLAKE, or paste your data-loading cell.


In [2]:
%pip -q install huggingface_hub

from huggingface_hub import hf_hub_download
import json

train_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="train.json",
    repo_type="dataset",
)

with open(train_path, "r", encoding="utf-8") as file:
    slake_train_data = json.load(file)

print("Loaded file:", train_path)
print("Top-level type:", type(slake_train_data).__name__)

if isinstance(slake_train_data, list):
    print("Number of training examples:", len(slake_train_data))
    sample = slake_train_data[0]

elif isinstance(slake_train_data, dict):
    print("Top-level keys:", list(slake_train_data.keys()))

    # Look for a list of examples inside the JSON object
    rows = next(
        (
            value for value in slake_train_data.values()
            if isinstance(value, list) and value
        ),
        None,
    )

    if rows is not None:
        print("Number of examples in first list:", len(rows))
        sample = rows[0]
    else:
        sample = slake_train_data

else:
    sample = slake_train_data

print("\nFirst example:")
print(json.dumps(sample, indent=2, ensure_ascii=False)[:3000])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


train.json:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

Loaded file: /root/.cache/huggingface/hub/datasets--BoKelvin--SLAKE/snapshots/a9083ce6c34ac3ffb17671a605962924d8a8f9e9/train.json
Top-level type: list
Number of training examples: 9835

First example:
{
  "img_id": 1,
  "img_name": "xmlab1/source.jpg",
  "question": "What modality is used to take this image?",
  "answer": "MRI",
  "q_lang": "en",
  "location": "Abdomen",
  "modality": "MRI",
  "answer_type": "OPEN",
  "base_type": "vqa",
  "content_type": "Modality",
  "triple": [
    "vhead",
    "_",
    "_"
  ],
  "qid": 0
}


In [3]:
from huggingface_hub import hf_hub_download
from zipfile import ZipFile
import json

image_folder = slake_train_data[0]["img_name"].split("/")[0]
print("Image folder:", image_folder)

imgs_zip_path = hf_hub_download(
    repo_id="BoKelvin/SLAKE",
    filename="imgs.zip",
    repo_type="dataset",
)

with ZipFile(imgs_zip_path) as archive:
    matching_files = []

    for path in archive.namelist():
        normalized_path = path.replace("\\", "/").strip("/")
        parts = normalized_path.split("/")

        if image_folder in parts:
            matching_files.append(normalized_path)

    print("\nFiles for this image:")
    for path in matching_files:
        print(" ", path)

    detection_files = [
        path for path in matching_files
        if path.lower().endswith("/detection.json")
        or path.lower() == "detection.json"
    ]

    if detection_files:
        detection_data = json.loads(archive.read(detection_files[0]))
        print("\nDetection annotation:")
        print(json.dumps(detection_data, indent=2, ensure_ascii=False)[:3000])
    else:
        print("\nNo detection.json found for this image folder.")

Image folder: xmlab1


imgs.zip: reconstructing file:   0%|          |  0.00B /  212MB            

imgs.zip: downloading bytes:           |  0.00B            


Files for this image:
  imgs/xmlab1
  imgs/xmlab1/question.json
  imgs/xmlab1/source.jpg
  imgs/xmlab1/mask.png
  imgs/xmlab1/detection.json

Detection annotation:
[
  {
    "Liver": [
      54.0,
      106.0,
      30.0,
      31.0
    ]
  }
]


In [4]:
from zipfile import ZipFile
from io import BytesIO
from PIL import Image

first_row = slake_train_data[0]
image_id = first_row["img_id"]
image_name = first_row["img_name"]

print("Image ID:", image_id)
print("Image name:", image_name)

print("\nQuestions for this image:")
for row in slake_train_data:
    if row.get("img_id") == image_id and row.get("q_lang") == "en":
        print(
            f"QID {row.get('qid')}: "
            f"{row.get('question')} "
            f"-> {row.get('answer')} "
            f"[{row.get('content_type')}]"
        )

source_file = next(
    (path for path in matching_files if path.lower().endswith("/source.jpg")),
    None,
)

if source_file:
    with ZipFile(imgs_zip_path) as archive:
        image = Image.open(BytesIO(archive.read(source_file)))
        print("\nImage dimensions (width, height):", image.size)
else:
    print("\nCould not find source.jpg in the matching files.")

Image ID: 1
Image name: xmlab1/source.jpg

Questions for this image:
QID 0: What modality is used to take this image? -> MRI [Modality]
QID 1: Which part of the body does this image belong to? -> Abdomen [Position]
QID 2: What is the mr weighting in this image? -> T2 [Modality]
QID 3: Does the picture contain liver? -> Yes [Organ]
QID 4: Does the picture contain kidney? -> No [Organ]
QID 5: Does the picture contain spleen? -> No [Organ]
QID 6: Where is the liver? -> Right [Position]

Image dimensions (width, height): (256, 256)


In [5]:
import json
from zipfile import ZipFile
from pathlib import PurePosixPath

image_name = "xmlab1/source.jpg"
image_folder = PurePosixPath(image_name).parent.as_posix()

# Find this image's detection.json inside the downloaded imgs.zip
with ZipFile(imgs_zip_path) as archive:
    detection_path = next(
        name for name in archive.namelist()
        if name.replace("\\", "/").endswith(
            f"{image_folder}/detection.json"
        )
    )
    detection_data = json.loads(archive.read(detection_path))

# Extract the Liver box from the annotation
liver_box = None
for item in detection_data:
    if "Liver" in item:
        liver_box = item["Liver"]
        break

# Make one label record per English question for this image
image_questions = [
    row for row in slake_train_data
    if row.get("img_name") == image_name
    and row.get("q_lang") == "en"
]

labels = []

for row in image_questions:
    question = row["question"].lower()
    answer = row["answer"].lower()

    asks_about_liver = "liver" in question
    has_positive_liver_answer = answer in {"yes", "right", "left"}

    use_liver_box = (
        asks_about_liver
        and has_positive_liver_answer
        and liver_box is not None
    )

    labels.append({
        "img_id": row["img_id"],
        "img_name": row["img_name"],
        "qid": row["qid"],
        "question": row["question"],
        "answer": row["answer"],
        "bbox_xywh_pixels": liver_box if use_liver_box else None,
        "grounding_status": (
            "grounded"
            if use_liver_box
            else "no_positive_region"
            if "does the picture contain" in question and answer == "no"
            else "not_required"
        )
    })

print(json.dumps(labels, indent=2))

[
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 0,
    "question": "What modality is used to take this image?",
    "answer": "MRI",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 1,
    "question": "Which part of the body does this image belong to?",
    "answer": "Abdomen",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 2,
    "question": "What is the mr weighting in this image?",
    "answer": "T2",
    "bbox_xywh_pixels": null,
    "grounding_status": "not_required"
  },
  {
    "img_id": 1,
    "img_name": "xmlab1/source.jpg",
    "qid": 3,
    "question": "Does the picture contain liver?",
    "answer": "Yes",
    "bbox_xywh_pixels": [
      54.0,
      106.0,
      30.0,
      31.0
    ],
    "grounding_status": "grounded"
  },
  {
    "img_id": 1,
    "img_name": "xm

In [6]:
for item in labels:
    print(
        f"QID {item['qid']}: answer={item['answer']}, "
        f"status={item['grounding_status']}, "
        f"box={item['bbox_xywh_pixels']}"
    )

QID 0: answer=MRI, status=not_required, box=None
QID 1: answer=Abdomen, status=not_required, box=None
QID 2: answer=T2, status=not_required, box=None
QID 3: answer=Yes, status=grounded, box=[54.0, 106.0, 30.0, 31.0]
QID 4: answer=No, status=no_positive_region, box=None
QID 5: answer=No, status=no_positive_region, box=None
QID 6: answer=Right, status=grounded, box=[54.0, 106.0, 30.0, 31.0]


In [7]:
from zipfile import ZipFile
from pathlib import PurePosixPath

english_rows = [
    row for row in slake_train_data
    if row.get("q_lang") == "en"
]

with ZipFile(imgs_zip_path) as archive:
    files = [
        name.replace("\\", "/").lstrip("./")
        for name in archive.namelist()
    ]

    source_files = {
        name.removeprefix("imgs/")
        for name in files
        if name.endswith("/source.jpg")
    }

    detection_folders = {
        str(PurePosixPath(name).parent).removeprefix("imgs/")
        for name in files
        if name.endswith("/detection.json")
    }

question_images = {
    row["img_name"]
    for row in english_rows
}

question_folders = {
    PurePosixPath(name).parent.as_posix()
    for name in question_images
}

print("English training questions:", len(english_rows))
print("Unique images with English questions:", len(question_images))
print("Question images found in archive:", len(question_images & source_files))
print("Question image folders with detection.json:", len(question_folders & detection_folders))
print("Example question image paths:", sorted(question_images)[:5])

English training questions: 4919
Unique images with English questions: 450
Question images found in archive: 450
Question image folders with detection.json: 450
Example question image paths: ['xmlab1/source.jpg', 'xmlab100/source.jpg', 'xmlab101/source.jpg', 'xmlab104/source.jpg', 'xmlab107/source.jpg']


In [8]:
import json
import os
import re
from pathlib import PurePosixPath
from zipfile import ZipFile

english_rows = [
    row for row in slake_train_data
    if row.get("q_lang") == "en"
]

training_records = []
grounded_count = 0
missing_images = []

with ZipFile(imgs_zip_path) as archive:
    archive_files = [
        name.replace("\\", "/")
        for name in archive.namelist()
    ]

    for row in english_rows:
        image_name = row["img_name"]
        folder = PurePosixPath(image_name).parent.as_posix()

        image_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + image_name)),
            None
        )
        detection_zip_path = next(
            (name for name in archive_files
             if name.endswith("/" + folder + "/detection.json")),
            None
        )

        if image_zip_path is None:
            missing_images.append(image_name)
            continue

        # Extract the image so the training dataset can load it by path.
        image_path = os.path.join("/content/slake_images", image_name)
        os.makedirs(os.path.dirname(image_path), exist_ok=True)

        if not os.path.exists(image_path):
            with open(image_path, "wb") as image_file:
                image_file.write(archive.read(image_zip_path))

        # Read this image's organ boxes.
        detections = []
        if detection_zip_path is not None:
            detections = json.loads(archive.read(detection_zip_path))

        question = row["question"].lower()
        answer = row["answer"].strip().lower()
        bbox = None
        matched_organ = None

        # Attach a box only when the question names that organ and
        # the answer indicates it is present or asks where it is.
        for item in detections:
            for organ, box in item.items():
                organ_words = re.findall(r"[a-z]+", organ.lower())
                question_words = re.findall(r"[a-z]+", question)

                if organ_words and all(word in question_words for word in organ_words):
                    asks_presence = any(
                        word in question
                        for word in ["contain", "present", "visible", "is there"]
                    )
                    asks_location = any(
                        word in question
                        for word in ["where", "which side", "position"]
                    )

                    positive_answer = answer in {
                        "yes", "right", "left", "present"
                    }

                    if positive_answer and (asks_presence or asks_location):
                        bbox = box
                        matched_organ = organ
                        break
            if bbox is not None:
                break

        if bbox is not None:
            grounded_count += 1

        training_records.append({
            "img_id": row["img_id"],
            "img_name": image_name,
            "image_path": image_path,
            "qid": row["qid"],
            "question": row["question"],
            "answer": row["answer"],
            "bbox_xywh_pixels": bbox,
            "grounded_organ": matched_organ,
        })

print("Training records created:", len(training_records))
print("Records with a matched box:", grounded_count)
print("Missing images:", len(missing_images))
print("Example record:", json.dumps(training_records[0], indent=2))

Training records created: 4919
Records with a matched box: 174
Missing images: 0
Example record: {
  "img_id": 1,
  "img_name": "xmlab1/source.jpg",
  "image_path": "/content/slake_images/xmlab1/source.jpg",
  "qid": 0,
  "question": "What modality is used to take this image?",
  "answer": "MRI",
  "bbox_xywh_pixels": null,
  "grounded_organ": null
}


In [9]:
from collections import Counter
from PIL import Image

grounded_records = [
    row for row in training_records
    if row["bbox_xywh_pixels"] is not None
]

print("Grounded examples:")
for row in grounded_records[:5]:
    print(
        f"QID {row['qid']}: {row['question']} "
        f"-> {row['answer']} | organ={row['grounded_organ']} "
        f"| box={row['bbox_xywh_pixels']}"
    )

valid_boxes = 0
invalid_boxes = []

for row in grounded_records:
    x, y, width, height = row["bbox_xywh_pixels"]

    with Image.open(row["image_path"]) as image:
        image_width, image_height = image.size

    is_valid = (
        x >= 0 and y >= 0 and width > 0 and height > 0
        and x + width <= image_width
        and y + height <= image_height
    )

    if is_valid:
        valid_boxes += 1
    else:
        invalid_boxes.append(row["img_name"])

print("\nGrounded records:", len(grounded_records))
print("Boxes inside image boundaries:", valid_boxes)
print("Invalid boxes:", len(invalid_boxes))
print("Grounded organ counts:", dict(Counter(
    row["grounded_organ"] for row in grounded_records
)))

Grounded examples:
QID 3: Does the picture contain liver? -> Yes | organ=Liver | box=[54.0, 106.0, 30.0, 31.0]
QID 6: Where is the liver? -> Right | organ=Liver | box=[54.0, 106.0, 30.0, 31.0]
QID 68: Does the picture contain liver? -> Yes | organ=Liver | box=[50.0, 102.0, 109.0, 151.0]
QID 70: Does the picture contain spleen? -> Yes | organ=Spleen | box=[190.0, 175.0, 75.0, 69.0]
QID 244: Does the picture contain liver? -> Yes | organ=Liver | box=[48.0, 141.0, 55.0, 106.0]

Grounded records: 174
Boxes inside image boundaries: 174
Invalid boxes: 0
Grounded organ counts: {'Liver': 119, 'Spleen': 20, 'Brain Stem': 1, 'Spinal Cord': 11, 'Esophagus': 2, 'Brain Edema': 1, 'Stomach': 3, 'Small Bowel': 7, 'Heart': 2, 'Bladder': 1, 'Colon': 7}
